---
title: "Projeto Integrador — Computação Para Ciência de Dados"
author: "Trabalho em grupo — PADS Insper"
---

**Objetivo:** Desenvolver um modelo preditivo para indicar se uma empresa deixará de operar em um período de até dois anos.


**Resultados**
- Este notebook implementa um pipeline completo de pré-processamento de dados para predizer se uma empresa irá **deixar de operar em até dois anos**.
- Os dados são da Bisnode (empresa europeia de business information), cobrindo os anos de 2005 a 2016.
- O output final deste arquivo é um csv tratado, que será utilizado para desenvolver um modelo preditivo para indicar se uma empresa deixará de operar em um período de até dois anos (considerando dados de 2012).


# Imports

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt
import dfply as dp
import missingno as msno


alt.data_transformers.disable_max_rows()

# Carregamento dos dados

O dataset principal é o `cs_bisnode_panel.csv`, com dados em painel de empresas entre 2005 e 2016.
Também carregamos o dicionário de variáveis para referência.

In [ ]:
df = pd.read_csv('cs_bisnode_panel.csv')

print(f'Shape: {df.shape}')
df.info()

In [ ]:
df.head()

In [ ]:
dic = pd.read_excel('bisnode_variable_names (2).xls')

dic

## Overview das variáveis

Antes de iniciar qualquer tratamento nos dados, fizemos um mapeamento simples das métricas, segmentando-as em grupos

**Variáveis descritivas**

*CEO Related*
- ceo_count
- foreign 
- female 
- birth_year
- inoffice_days
- gender
- origin


*Company Related*
- exit_date + exit_year
- founded_date
- nace_main
- ind2
- ind 
- urban_m
- region_m
- labor_avg

**Variáveis numéricas**

*Inflows*
- sales
- extra_inc

*Outflows*
- personnel_exp
- extra_exp
- material_exp
- amort

*Assets*
- intang_assets
- tang_assets
- liq_assets
- fixed_assets
- curr_assets
- inventories
- subscribed_cap

*Liabilities*
- curr_liab

*Resultados* (podem ser negativos)
- share_eq
- profit_loss_year
- inc_bef_tax


# Limpeza dos Dados

## Requirements do Projeto

O primeiro passo na limpeza dos dados foi executar as filtragens básicas, delimitadas no escopo do projeto

- Remover registros de 2016
- Filtrar apenas dados de 2012
- Receitas entre 1000 Euros e 10MM Euros
- Remover as colunas ['COGS', 'finished_prod', 'net_dom_sales', 'net_exp_sales', 'wages', 'D'] (alta missing rate)


Extraímos o ano a partir da coluna `begin` (data de início do período de referência).

O ano de 2016 é removido pois não temos dados suficientes para calcular o default (precisamos de X+2 anos).

In [ ]:
df['year'] = pd.to_datetime(df['begin']).dt.year
df = df[df['year'] != 2016]
df['year'].value_counts().sort_index()

As colunas abaixo apresentam mais de 93% de dados faltantes, tornando-as inviáveis para modelagem:
- `D`: 100% missing
- `finished_prod`, `wages`, `COGS`, `net_exp_sales`, `net_dom_sales`: ~94% missing

Limpeza de colunas redundantes
- Removemos colunas de data (`begin`, `end`) que foram substituídas por `year`.
- Removemos as colunas `exit_date` e `exit_year`, pois se referem ao ano de fim de operação, o que representaria data leakage para o nosso modelo

In [ ]:
df = df.drop(columns=['D', 'finished_prod', 'wages', 'COGS', 'net_exp_sales', 'net_dom_sales', 'begin', 'end', 'exit_date', 'exit_year'])
print(f'Shape após drop: {df.shape}')

Mantemos apenas empresas com faturamento entre **1.000€ e 10.000.000€**, removendo micro-registros e grandes grupos que já foram excluídos pela Bisnode.

In [ ]:
df = df[
    (df['sales'] >= 1000) &
    (df['sales'] <= 10000000)
]
print(f'Shape após filtro de receita: {df.shape}')
df['sales'].describe()

Filtrar para empresas criadas em 2012 - um dos requisitos do projeto.

In [ ]:
df_filtered_2012 = df[df['year'] == 2012].copy()
print(f'Shape df_filtered: {df_filtered_2012.shape}')

In [ ]:
df_filtered_2012.describe()

## Tratamento de Inconsistências

Além da limpeza da base para os requirements básicos do projeto, tratamos algumas inconsistências para garantir dados consistentes para os posteriores modelos, como valores negativos em algumas variáveis financeiras referentes aos balanços das empresas avaliadas.

Por definição econômica, dentre as variáveis numéricas que estão na nossa base, as únicas variáveis que **podem ser negativas** são as relacionadas a resultados:

- share_eq
- profit_loss_year
- inc_bef_tax

O resto, caso negativo, foi considerado como zero.
Ao validarmos a quantidade de linhas por métrica que foi zerada dados valores negativos, encontramos um % de linhas baixo comparado ao total da base, o que nos faz concluir que estes negativos eram de fato algumas inconsistências residuais.

In [ ]:
cols_nao_negativas = [
    'sales', 'extra_inc'
    'personnel_exp', 'material_exp', 'extra_exp','amort', 'curr_liab',
    'intang_assets', 'tang_assets', 'liq_assets', 'fixed_assets', 'curr_assets', 'inventories', 'subscribed_cap'
]

for col in cols_nao_negativas:
    if col in df_filtered_2012.columns:
        n_neg = (df_filtered_2012[col] < 0).sum()
        if n_neg > 0:
            print(f'{col}: {n_neg} valores negativos → zerados')
        df_filtered_2012[col] = df_filtered_2012[col].clip(lower=0)

print('\nVerificação final:')
for col in cols_nao_negativas:
    if col in df_filtered_2012.columns:
        assert df_filtered_2012[col].min() >= 0, f'ERRO: {col} ainda tem negativos!'
print('Nenhum valor negativo inválido encontrado.')

## Análise de Missing Values

Além dos dados que já retiramos, conduzimos uma análise subsequente para identificar outros dados faltantes e tratá-los, utilizando o `missingno` para visualização.

Primeiramente, apenas mapeamos as métricas com mais missing rate para visualizar os principais problemas da nossa base que precisarão ser tratados.
As metricas mais críticas foram:

- birth_year
- labor_avg
- founded_year

In [ ]:
threshold = 0.02
cols_com_nulos = df_filtered_2012.columns[df_filtered_2012.isnull().mean() > threshold]

msno.bar(df_filtered_2012[cols_com_nulos], sort="descending", color="steelblue", figsize=(10, 5), labels=True)

Gerando uma matriz de missing values e dendograma, percebemos alguns padrões que nos ajudam a definir o tratamento de missing values:

1) As colunas `ceo_count`, `foreign`, `female`, `inoffice_days`, `gender`, `origin` tem exatamente os mesmos cortes de nulos; a falta destas deve estar relacionada a uma mesma causa. Dessa forma, trataremos todas da mesma forma para manter consistência.
2) A coluna `founded_year`, apesar de ter linhas faltantes muitas vezes coincidente com as outras metricas de CEO, não está 100% relacionada;
3) `ind`, `labor_avg` e `birth_year` tem cortes completamente distintos

In [ ]:
msno.matrix(
    df_filtered_2012[cols_com_nulos].sample(500),
    figsize=(10, 5),
    fontsize=10,
    color=(0.25, 0.25, 0.5)
)


In [ ]:
msno.dendrogram(df_filtered_2012[cols_com_nulos], figsize=(8, 5), 
    fontsize=10, orientation='left')

**founded_year**

Pontos de validação:
- Existe uma coluna chamada `founded_date`, que não aparece aqui, ou seja, tem menos nulos que `founded_year`. O que acontece?


Conclusão:
- Apesar de `founded_date` ser muito mais presente, quando fazemos uma validação das linhas onde ambas as colunas estão preenchidas, não existe nenhum mismatch
- Para "tratar" este nulo, vamos apenas gerar novamente o `founded_year` baseado na coluna de date.

In [ ]:
# Comparação dos nulos
print(df_filtered_2012[['founded_date', 'founded_year']].isna().sum())

# Casos onde uma existe e a outra não
conflito_nulos = df_filtered_2012[df_filtered_2012['founded_date'].isna() != df_filtered_2012['founded_year'].isna()]
print(f"Linhas com apenas um dos campos preenchidos: {len(conflito_nulos)}")

## Mismatch considerável entre ambas, dado que deveriam dar a mesma informação

In [ ]:
# Validar se, nos casos que temos informação, as duas batem (eu poderia resolver os nulos de founded_year com founded_date)
validacao_fy = df_filtered_2012.copy()

validacao_fy['founded_date_dt'] = pd.to_datetime(df_filtered_2012['founded_date'], errors='coerce')
validacao_fy['year_extracted'] = validacao_fy['founded_date_dt'].dt.year

divergentes = validacao_fy[
    (validacao_fy['year_extracted'].notna()) & 
    (validacao_fy['founded_year'].notna()) & 
    (validacao_fy['year_extracted'] != validacao_fy['founded_year'])
]

print(f"Casos onde o ano da data não bate com a coluna year: {len(divergentes)}")


## Dado o resultado, vamos usar a informação de fundação baseado na métrica mais presente, resolvendo os missing de founded_year

**labor_avg** 

Ponto de validação:
- Faz sentido fazer uma média por tipo de industria? precisa? se a média de `labor_avg` for similar independente do setor, usar a média global.

Conclusão:
- Temos uma variação considerável da coluna de `labor_avg` a depender do setor, então a estratégia para tratar os nulos será preencher com a uma média para cada `ind2`

In [ ]:
plt.figure(figsize=(12, 6))
# Filtrando apenas os 10 setores mais comuns para o gráfico não ficar muito confuso.
top_setores = df_filtered_2012['ind2'].value_counts().nlargest(10).index
df_plot = df_filtered_2012[df_filtered_2012['ind2'].isin(top_setores)]

sns.boxplot(data=df_plot, x='ind2', y='labor_avg')
plt.yscale('log') # Usando Log porque empresas variam muito de tamanho.
plt.title('Dispersão de labor_avg por Setor')
plt.show()

In [ ]:
media_global = df_filtered_2012['labor_avg'].mean()
medias_por_setor = df_filtered_2012.groupby('ind2')['labor_avg'].mean()

print(f"Média Global: {media_global:.2f}")
print("--- Variação por Setor ---")
print(medias_por_setor.describe())

## Variação considerável; usar setor parece fazer sentido para preencher essa feature

### Tratamento de Missing Values

Estratégia final por variável:
- `founded_year`: usaremos a métrica founded_year a partir da coluna founded_date, que tem baixa presença de nulos
- `labor_avg`: substituir pela média do mesmo setor (`ind2`) no dataset completo
- Colunas sobre CEO: `birth_year`, `gender`,  `female`, `foreign`, `origin`, `ceo_count`, `inoffice_days`: substituir pela média geral/moda - no caso de variáveis categóricas; como as linhas faltantes são as mesmas, trataremos com o mesmo método.
- Apesar de birth_year não estar diretamente relacionada em tipo de missing value com as outras informações sobre CEO, escolhemos tratá-la da mesma forma para manter consistência nessa categoria como um todo.

In [ ]:
def missing_treatment(df):
    df_clean = df.copy()
    
    # 1. founded_year (Usa founded_date e depois remove a data)
    df_clean['founded_date'] = pd.to_datetime(df_clean['founded_date'], errors='coerce')
    df_clean['founded_year'] = df_clean['founded_year'].fillna(df_clean['founded_date'].dt.year)
    df_clean = df_clean.drop(columns=['founded_date'], errors='ignore')
    
    # 2. labor_avg (Média por setor 'ind2')
    df_clean['labor_avg'] = df_clean.groupby('ind2')['labor_avg'].transform(
        lambda x: x.fillna(x.mean())
    )
    # Fallback para média global (caso o setor inteiro seja NaN)
    df_clean['labor_avg'] = df_clean['labor_avg'].fillna(df_clean['labor_avg'].mean())
    
    # 3. Colunas do CEO: Numéricas vs Categóricas
    cols_categoricas = ['gender', 'origin', 'female', 'foreign']
    cols_numericas = ['birth_year', 'ceo_count', 'inoffice_days']
    
    # 4. Tratamento Categóricas (Moda)
    for col in cols_categoricas:
        if col in df_clean.columns:
            moda = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(moda)
            
    # Tratamento Numéricas (Média)
    for col in cols_numericas:
        if col in df_clean.columns:
            media = df_clean[col].mean()
            df_clean[col] = df_clean[col].fillna(media)
    
    return df_clean

# Executando a limpeza
df_clean = missing_treatment(df_filtered_2012)

In [ ]:
print('Missing values restantes:', df_clean.isnull().sum().sum())

In [ ]:
df_clean[['urban_m', 'region_m', 'ind', 'ind2']]

Após o tratamento de nulos principais, ainda temos colunas com alguns dados faltantes, então fazemos um segundo processamento:

- **Variáveis financeiras**: Assumimos que, caso esteja nula, significa zero, então este foi o procedimento adotado;
- **Métricas "básicas" sobre a empresa**: as informações de founded_year (baseada na founded_date - tratada na primeira leva de missing values treatment), ind2 e nace_main, por trazerem informações muito cruciais sobre cada empresa (início da operação e indústria) e por possuírem apenas 3/5 linhas faltantes, serão excluídas. Além do prejuízo de dados perdidos ser pequeno, não queremos fazer nenhuma assumption específica em cima dos dados faltantes;
- **Categóricas restantes**: para `region_m` e `ind`, utilizamos a moda. A `ind` especificamente possui um volume mais considerável de dados faltantes, no entanto não escolhemos fazer nenhum processamento muito profundo porque ela possui um dado muito similar à `ind2`, e ao avaliar ambas as métricas (na sessão de engenharia de features) escolhemos manter apenas a segunda.

In [ ]:
nulos_restantes = df_clean.isna().sum()
print(nulos_restantes[nulos_restantes > 0])

In [ ]:
def missing_treatment_part2(df):
    df_clean = df.copy()
    
    # 1. Remover linhas essenciais
    # Se não tem founded_year, ind2 ou nace_main, a linha é descartada. São poucas observações que perdemos
    cols_essenciais = ['founded_year', 'ind2', 'nace_main']
    df_clean = df_clean.dropna(subset=cols_essenciais)
    
    # 2. Variáveis Financeiras(NaN -> 0)
    # Assumimos que a ausência de registro financeiro significa valor zero.
    cols_financeiras = [
        'amort', 'curr_assets', 'curr_liab', 'fixed_assets', 'intang_assets',
        'inventories', 'liq_assets', 'material_exp', 'personnel_exp',
        'profit_loss_year', 'share_eq', 'subscribed_cap', 'tang_assets'
    ]
    for col in cols_financeiras:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna(0)
            
    # 3. Região (Moda)
    if 'region_m' in df_clean.columns:
        moda_regiao = df_clean['region_m'].mode()[0]
        df_clean['region_m'] = df_clean['region_m'].fillna(moda_regiao)
        
    # 4. Ind (Broad Industry Code)
    # Dado que o Ind2, que é mais detalhadado, já foi tratado, possivelmente nem usaremos esta coluna como feature (pode ser redundante). Por hora, vamos usar a moda também
    if 'ind' in df_clean.columns:
        moda_ind = df_clean['ind'].mode()[0]
        df_clean['ind'] = df_clean['ind'].fillna(moda_ind)
    
    return df_clean

df_clean_final = missing_treatment_part2(df_clean)

# Validação final de nulos
print(f"Nulos restantes: {df_clean_final.isnull().sum().sum()}")

# Target Variable

Criação da variável dependente: `default`

**Definição:** Uma empresa deu default se esteve ativa no ano X, mas **não apresentou vendas nos 2 anos seguintes (X+1 e X+2)**, sem retomar faturamento depois.

**Lógica implementada:**
1. Fazemos um pivot (`unstack`) para ter as empresas nas linhas e os anos nas colunas
2. Percorremos a série temporal de cada empresa de trás para frente
3. Se os últimos 2+ anos forem `NaN` sem nenhum número após eles → `default = 1`
4. Se houver retomada de faturamento (NaN no meio mas número depois) → `default = 0`

In [ ]:
# Pivot: linhas = empresa, colunas = ano
sales_year = df.groupby(['comp_id', 'year'])['sales'].sum().unstack()
sales_year.head()

In [ ]:
def empresa_saiu(row):
    """Retorna 1 se a empresa tem 2+ NaNs consecutivos no final da série temporal."""
    values = row.values
    count = 0
    for v in reversed(values):
        if pd.isna(v):
            count += 1
        else:
            break 
    return 1 if count >= 2 else 0

default_series = sales_year.apply(empresa_saiu, axis=1).reset_index(name='default')
default_series = default_series[['comp_id', 'default']]

In [ ]:
# Sanity Check

default_series[default_series.comp_id.isin([1003200, 1002029, 1018301])] # first should be 1, second should be 0, third 0

In [ ]:
df_clean_final_w_target = pd.merge(df_clean_final, default_series, on='comp_id', how='left')

In [ ]:
# Garante que não há coluna default duplicada antes do merge

print('Distribuição do default:')
print(df_clean_final_w_target['default'].value_counts())
print(f'\nTaxa de default: {df_clean_final_w_target["default"].mean():.1%}')

 **Gráfico: Distribuição da variável target**

 Quando avaliamos a distribuição do nosso target, podemos ver que a distribuição não é proporcional. Isso já nos indica que temos que tomar alguns cuidados quando formos fazer os modelos, como garantir uma proporção equivalente de default vs. não default no treino e teste, por exemplo.

In [ ]:
default_counts = df_clean_final_w_target.drop_duplicates('comp_id')['default'].value_counts().reset_index()
default_counts.columns = ['default', 'count']
default_counts['label'] = default_counts['default'].map({0: 'Ativa', 1: 'Deixou de operar'})

alt.Chart(default_counts).mark_bar().encode(
    x=alt.X('label:N', title='Status'),
    y=alt.Y('count:Q', title='Nº de Empresas'),
    color=alt.Color('label:N', scale=alt.Scale(range=['steelblue', 'crimson'])),
    tooltip=['label', 'count']
).properties(title='Distribuição da Variável Target (default) - 2012', width=300)

In [ ]:
df_chart = df_clean_final_w_target[['sales', 'default']].copy()
df_chart['default_label'] = df_chart['default'].map({0: 'Ativa', 1: 'Deixou de operar'})

alt.Chart(df_chart).mark_boxplot(extent='min-max').encode(
    x=alt.X('default_label:N', title='Status'),
    y=alt.Y('sales:Q', title='Vendas (€)', scale=alt.Scale(type='log')),
    color=alt.Color('default_label:N', scale=alt.Scale(range=['steelblue', 'crimson']))
).properties(title='Distribuição de Vendas por Status (escala log)', width=300)

# Engenharia de features

**Resumo:**

  - `idade_empresa`: calculada como a diferença entre o ano de referência (2012) e `founded_year`.
  - Remoção de `gender` e `origin`, redundantes com as variáveis binárias `female` e `foreign`.
  - Seleção de `ind2` e remoção de `ind` por alta colinearidade entre ambas.
  - Criação de `equity_ratio` (shareholder equity / ativos totais) e `fixed_asset_ratio` (ativos fixos / ativos totais) e `capitalization_ratio` (subscribed capital / ativos totais) a partir dos componentes do balanço.
  - Remoção de `balsheet_flag` e `balsheet_length` por variância quase zero e redundância.
  - Criação de `sales_growth` (crescimento percentual de vendas entre 2011 e 2012), `was_operating_2011` (indicador de operação em 2011) e `is_growing` (indicador binário de crescimento positivo).
  - Transformação logarítmica (`log1p`) aplicada a 16 variáveis com assimetria (skew) superior a 5, incluindo `sales`, `curr_assets`, `fixed_assets`, `personnel_exp`, `amort`, `extra_inc`, `extra_exp`, `curr_liab`, `intang_assets`, `liq_assets`, `material_exp`, entre outras.
  - Agrupamento de categorias raras (prevalência < 1%) em variáveis categóricas, seguido de One-Hot Encoding.
  - Remoção de colunas com variância quase zero (> 98% dos valores iguais).

## Features Overview

### Principais Insights do Features Overview

A análise exploratória inicial das variáveis disponíveis revela padrões importantes que orientam as etapas de feature engineering a seguir:

1. **Distribuições assimétricas nas variáveis financeiras:** Variáveis como vendas (`sales`), despesas (`expenses`) e ativos (`assets`) apresentam distribuições com cauda longa à direita (right-skewed), com grande concentracao em valores baixos e poucas empresas com valores extremamente elevados. Isso indica necessidade de a aplicação de transformações logarítmicas (`log1p`) nas etapas seguintes, a fim de reduzir a assimetria e melhorar o desempenho dos modelos preditivos.

2. **Diferenças visíveis entre empresas ativas e default:** Em diversas métricas financeiras, observam-se diferenças nas distribuições entre os grupos `default=0` (empresas ativas) e `default=1` (deixaram de operar). Isso valida o potencial preditivo dessas features e justifica sua inclusão no modelo.

3. **Variáveis redundantes ou relacionadas:** Algumas variáveis possuem alta colinearidade ou são derivações umas das outras (por exemplo, `ind` e `ind2`, `gender` e `female`, `origin` e `foreign`). Isso motiva as etapas de seleção e redução de features para evitar multicolinearidade e simplificar o modelo.

4. **Categorias raras em variáveis categóricas:** Variáveis como `ind2`, `nace_main` e `region_m` apresentam diversas categorias com frequência muito baixa. Isso justifica o agrupamento de categorias raras (com prevalência < 1%) em uma categoria única antes da aplicação de One-Hot Encoding, evitando assim a criação de features esparsas e pouco informativas.

In [ ]:
mapping = {
    # Inflows
    'sales': '1. Inflows: Sales',
    'extra_inc': '1. Inflows: Extra Income',
    # Outflows
    'personnel_exp': '2. Outflows: Personnel Exp',
    'extra_exp': '2. Outflows: Extra Exp',
    'material_exp': '2. Outflows: Material Exp',
    'amort': '2. Outflows: Amortization',
    'curr_liab': '2. Outflows: Current Liabilities',
    # Assets
    'intang_assets': '3. Assets: Intangible',
    'tang_assets': '3. Assets: Tangible',
    'liq_assets': '3. Assets: Liquid',
    'fixed_assets': '3. Assets: Fixed',
    'curr_assets': '3. Assets: Current',
    'inventories': '3. Assets: Inventories',
    # Results
    'share_eq': '4. Results: Share Equity',
    'profit_loss_year': '4. Results: Profit/Loss Year',
    'inc_bef_tax': '4. Results: Income Before Tax',
    # CEO Related - Numeric
    'ceo_count': '5. CEO: Count',
    'female': '5. CEO: Female Presence',
    'inoffice_days': '5. CEO: In Office Days',
    'labor_avg': '6. Company: Labor Avg',
}

In [ ]:
df_long = df_clean_final_w_target.melt(
    id_vars=['default'], 
    value_vars=list(mapping.keys()), 
    var_name='Metrica_Tecnica', 
    value_name='Valor'
)

df_long['Metrica_Grupo'] = df_long['Metrica_Tecnica'].map(mapping)

# Drop
opcoes_ordenadas = sorted(list(mapping.values()))
input_dropdown = alt.binding_select(options=opcoes_ordenadas, name='Grupo e Métrica: ')
selection = alt.selection_point(fields=['Metrica_Grupo'], bind=input_dropdown, value=opcoes_ordenadas[0])

chart = alt.Chart(df_long).mark_bar(opacity=0.6).encode(
    alt.X("Valor:Q", bin=alt.Bin(maxbins=40), title="Valor Nominal"),
    alt.Y("count()", stack=None, title="Frequência"),
    alt.Color("default:N", scale=alt.Scale(range=['#4682b4', '#d62728'])),
    alt.Row("default:N")
).add_params(
    selection
).transform_filter(
    selection
).properties(width=500, height=150)

chart.display()

In [ ]:
df_long = df_clean_final_w_target.melt(
    id_vars=['default'], 
    value_vars=list(mapping.keys()), 
    var_name='Metrica_Tecnica', 
    value_name='Valor'
)

df_long['Metrica_Grupo'] = df_long['Metrica_Tecnica'].map(mapping)

df_long['Log_Valor'] = np.log1p(df_long['Valor'].clip(lower=0))

opcoes_ordenadas = sorted(list(mapping.values()))
input_dropdown = alt.binding_select(options=opcoes_ordenadas, name='Grupo e Métrica: ')
selection = alt.selection_point(fields=['Metrica_Grupo'], bind=input_dropdown, value=opcoes_ordenadas[0])

chart = alt.Chart(df_long).mark_bar(opacity=0.6).encode(
    alt.X("Log_Valor:Q", bin=alt.Bin(maxbins=40), title="Escala Logarítmica"),
    alt.Y("count()", stack=None, title="Frequência"),
    alt.Color("default:N", scale=alt.Scale(range=['#4682b4', '#d62728'])),
    alt.Row("default:N")
).add_params(
    selection
).transform_filter(
    selection
).properties(width=500, height=150)

chart.display()

## Métricas redundantes: Gender vs Female / Origin vs. Foreign

In [ ]:
df_filtered_2012['origin'].value_counts()

In [ ]:
df_clean_final_w_target[['gender', 'female', 'origin', 'foreign', 'ind', 'ind2']]

In [ ]:
df_analise = df_clean_final_w_target[['gender', 'female', 'origin', 'foreign']].copy()


mapping_gender = {'male': 0, 'female': 1, 'mix': 0.5}
mapping_origin = {'Domestic': 0, 'Foreign': 1, 'mix': 0.5} 

df_analise['gender_num'] = df_analise['gender'].map(mapping_gender)
df_analise['origin_num'] = df_analise['origin'].map(mapping_origin)

corr_gender = df_analise['gender_num'].corr(df_analise['female'])
corr_origin = df_analise['origin_num'].corr(df_analise['foreign'])

print(f"--- Resultado da Análise de Redundância ---")
print(f"Correlação entre 'gender' e 'female': {corr_gender:.4f}")
print(f"Correlação entre 'origin' e 'foreign': {corr_origin:.4f}")

inconsistencias = df_analise[
    ((df_analise['gender'] == 'male') & (df_analise['female'] > 0)) |
    ((df_analise['gender'] == 'female') & (df_analise['female'] < 1))
]

print(f"\nTotal de linhas com possível divergência de lógica: {len(inconsistencias)}")

Como as métricas são 100% redundantes, vamos usar apenas as métricas que já **são numéricas (female e foreign)**

In [ ]:
df_clean_final_w_target = df_clean_final_w_target.drop(columns=['gender', 'origin'], errors='ignore')

## Hierarquia: Ind vs. Ind2 vs Nace

In [ ]:
chart = alt.Chart(df_clean_final_w_target).mark_rect().encode(
    x=alt.X('ind2:O', title='Nace Industry Code (ind2)'),
    y=alt.Y('ind:O', title='Broad Industry Code (ind)'),
    color=alt.Color('count():Q', scale=alt.Scale(scheme='viridis'), title='Frequência'),
    tooltip=['ind', 'ind2', 'count()']
).properties(
    width=700,
    height=400,
    title='Hierarquia: ind vs ind2'
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
)

chart

A relação entre as duas features é hierárquica e mutuamente exclusiva (a feature ind "contém" perfeitamente a ind2); a feature ind2 me traz mais granularidade vs. a coluna ind.

**Vamos manter apenas a ind2**, pois ela traz mais informação e, se usássemos as duas, teriamos duas variáveis colineares nos modelos.

In [ ]:
chart = alt.Chart(df_clean_final_w_target).mark_rect().encode(
    x=alt.X('ind2:O', title='Nace Industry Code (ind2)'),
    y=alt.Y('nace_main:O', title='Nace Industry Code (nace_main)'),
    color=alt.Color('count():Q', scale=alt.Scale(scheme='viridis'), title='Frequência'),
    tooltip=['nace_main', 'ind2', 'count()']
).properties(
    width=700,
    height=400,
    title='Hierarquia: nace_main vs ind2'
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
)

chart

A `nace_main` parece ser uma `ind2` mais granular - mas também altamente correlacionadas.

Ex.: para `ind2 = 47`, temos `nace_main = [4777, 4754, 4791,...]`

In [ ]:
# Calculando a taxa de default por nace_main
df_stats = df_clean_final_w_target.groupby(['ind2', 'nace_main'])['default'].mean().reset_index()

plt.figure(figsize=(15,6))
sns.boxplot(data=df_stats, x='ind2', y='default')
plt.title('Variância da Taxa de Default do nace_main dentro de cada ind2')
plt.xticks(rotation=45)
plt.show()

A `nace_main` parece ter uma variação considerável dentro de cada `ind_2`, então a princípio vamos mantê-la no dataset final. No entanto, temos muitos segmentos com pouquíssimas observações, então precisaremos fazer ainda algum tratamento (agrupar categorias raras) 

In [ ]:
# Verificando quantos grupos do nace_main são "pequenos demais"
contagem = df['nace_main'].value_counts()
print(f"Total de categorias: {len(contagem)}")
print(f"Categorias com menos de 30 empresas: {(contagem < 30).sum()}")

In [ ]:
df_clean_final_w_target = df_clean_final_w_target.drop(columns=['ind'], errors='ignore')

## Balance Sheet Columns

In [ ]:
cols_meta = ['balsheet_flag', 'balsheet_length', 'balsheet_notfullyear']

for col in cols_meta:
    print(f"\n{'='*10} Análise da coluna: {col} {'='*10}")
    
    print("Distribuição de valores (%):")

    
    # Usamos display para criar uma tabelinha formatada para cada coluna
    display(df_clean_final_w_target[col].value_counts(normalize=True).to_frame().head(10) * 100)
    
    print("Taxa de Default por categoria:")
    display(df_clean_final_w_target.groupby(col)['default'].mean().to_frame() * 100)

`balsheet_notfullyear` é a versão binária da `balsheet_length`; Para evitar os ruídos das classificações da length, vamos usar apenas a notfullyear;

`balsheet_flag` tem 99% dos dados em 0. Usar essa metrica poderia gerar ruídos desnecessários no fit dada a distribuição da sample

In [ ]:
df_clean_final_w_target = df_clean_final_w_target.drop(columns=['balsheet_flag', 'balsheet_length'], errors='ignore')

## Idade da empresa

Criamos a variável `idade_empresa` como a diferença entre o ano de referência e o ano de fundação (`founded_year`).

In [ ]:
df_clean_final_w_target['idade_empresa'] = df_clean_final_w_target['year'] - df_clean_final_w_target['founded_year']
df_clean_final_w_target['idade_empresa'] = df_clean_final_w_target['idade_empresa'].fillna(0)

df_clean_final_w_target['idade_empresa'].describe()
# Idades negativas?

In [ ]:
alt.Chart(df_clean_final_w_target[df_clean_final_w_target['idade_empresa'] > 0]).mark_bar().encode(
    x=alt.X('idade_empresa:Q', bin=alt.Bin(maxbins=30), title='Idade (anos)'),
    y=alt.Y('count():Q', title='Nº de Empresas'),
    color=alt.value('steelblue')
).properties(title='Distribuição da Idade das Empresas (2012)', width=500)

In [ ]:
# Verificando quem são as empresas "do futuro"
empresas_futuro = df_clean_final_w_target[df_clean_final_w_target['idade_empresa'] < 0]
empresas_futuro[['comp_id', 'year', 'founded_year', 'idade_empresa']]

print(empresas_futuro['comp_id'].astype('int64').values)

In [ ]:
# Sanity Check

df[df.comp_id.isin([24919462, 104528502784, 290620375040])][['comp_id','sales', 'year', 'founded_year', 'founded_date']] 

## Existe algum mismatch de ano de fundação. Apesar de haver sales antes, os anos de fundação só constam posteriormente. Talvez empresas recentes, que já possuem alguma informação de venda mas não uma data formal de fundação.
## Nossa decisão é considerar que, quando o resultado da idade for negativo, vamos considerar idade = 0.

In [ ]:
# Ajuste da idade: se for negativa, vira 0 (empresa no ano de nascimento)
df_clean_final_w_target['idade_empresa'] = (df_clean_final_w_target['year'] - df_clean_final_w_target['founded_year'])
df_clean_final_w_target.loc[df_clean_final_w_target['idade_empresa'] < 0, 'idade_empresa'] = 0

# Validando o resultado final
print("Distribuição Final da Idade:")
print(df_clean_final_w_target['idade_empresa'].describe())

## Assets e Ratios de Performance



Quando avaliamos correlação entre as colunas de ativos, a primeira coisa que chama atenção é a comparação entre `fixed_assets` e `tang_assets` (96% de correlação). Como ambas parecem ser basicamente a mesma coisa, vamos remover tang_assets e usar apenas `fixed_assets`

In [ ]:
cols_assets = ['intang_assets', 'tang_assets', 'liq_assets', 'fixed_assets', 'curr_assets', 'inventories']
corr_matrix = df_clean_final_w_target[cols_assets].corr().reset_index().melt('index')
corr_matrix.columns = ['var1', 'var2', 'correlation']

heatmap = alt.Chart(corr_matrix).mark_rect().encode(
    x=alt.X('var1:N', title=None),
    y=alt.Y('var2:N', title=None),
    color=alt.Color('correlation:Q', 
                    scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
                    title="Correlação"),
    tooltip=[
        alt.Tooltip('var1', title='Variável 1'),
        alt.Tooltip('var2', title='Variável 2'),
        alt.Tooltip('correlation', title='Correlação', format='.2f')
    ]
).properties(
    width=400,
    height=400,
    title='Validação de Hierarquia: Ativos'
)

text = heatmap.mark_text().encode(
    text=alt.Text('correlation:Q', format='.2f'),
    color=alt.condition(
        alt.datum.correlation > 0.5, 
        alt.value('white'), 
        alt.value('black')
    )
)

(heatmap + text).display()

**Total Assets**

A princípio, por definição, total assets deveriam ser: Current + Fixed Assets 

No entanto, ficamos em dúvida sobre ser ou não necessário incluir Intangible Assets. Fizemos uma validação para garantir que Intangible Assets deveriam entrar na lógica dos ativos totais.

Sendo a variação muito sútil entre ambas alternativas e a correlação de fixed assets e intangible assets baixa, assumimos que seria necessária a inclusão.

Dessa forma, para todos os cálculos de Total Assets seguimos a lógica:

$ \text{Total Assets} = \text{Current Assets} + \text{Fixed Assets} + \text{Intangible Assets} $

In [ ]:
ratios_analysis = df_clean_final_w_target.copy()


ratios_analysis['total_v1'] = ratios_analysis['curr_assets'] + ratios_analysis['fixed_assets']
ratios_analysis['total_v2'] = ratios_analysis['curr_assets'] + ratios_analysis['fixed_assets'] + ratios_analysis['intang_assets']

print("Resumo V1 (Curr + Fixed):")
print(ratios_analysis['total_v1'].describe())

print("\nResumo V2 (Curr + Fixed + Intangible):")
print(ratios_analysis['total_v2'].describe())

Para complementar as informações "cruas" numéricas que temos no dataset, adicionamos alguns ratios de performance. Nossa hipótese é que, para além da métrica base, se olharmos para ela como uma proporção do todo, podemos extrair informações mais robustas - e o mesmo deve acontecer para os nossos modelos.

**Fixed Asset Ratio**
Indica a proporção do Ativo Total que está comprometida em ativos de longo prazo (imobilizado). Reflete a intensidade de capital da operação.

$$\text{Fixed Asset Ratio} = \frac{\text{Fixed Assets}}{\text{Total Assets}}$$

**Equity Ratio**
Mede a participação do capital próprio no financiamento dos ativos. Indicador de solvência e autonomia financeira.

$$\text{Equity Ratio} = \frac{\text{Shareholder Equity}}{\text{Total Assets}}$$

**Capitalization Ratio**
Mede o peso do capital subscrito em relação ao ativo total. Ao contrário do Equity Ratio, este índice foca no aporte inicial dos shareholders

$$\text{Capitalization Ratio} = \frac{\text{Subscribed Capital}}{\text{Total Assets}}$$

**Liquidity Ratio**
Avalia a capacidade de honrar obrigações de curto prazo usando ativos líquidos.

$$\text{Liquidity Ratio} = \frac{\text{Liquid Assets}}{\text{Current Assets} + \epsilon}$$

**Inventory Ratio**
Mede o quanto do Ativo Circulante está retido em estoques, indicando a composição da liquidez de curto prazo.

$$\text{Inventory Ratio} = \frac{\text{Inventories}}{\text{Current Assets} + \epsilon}$$

---

Para garantir mais robustez estatística dos modelos que vamos utilizar:
1) Adicionamos um valor infinitesimal ($1e-6$) aos denominadores. Isso previne erros de indefinição matemática (`Inf` ou `NaN`) em empresas que possuem Ativo Circulante nulo.
2) Clipping: Todos os ratios foram limitados ao intervalo $[-2, 2]$. Esses ratios financeiros podem apresentar valores extremos, e assim evitamos outliers. No entanto, ainda que limitados a 2 por exemplo, um índice de 200% vai ser considerado muito alto, o que não deveria nos fazer perder muita interpretabilidade

In [ ]:
ratios_analysis = df_clean_final_w_target.copy()

# Epsilon só pra garantir que a base não vai ser zero e dar erro na divisão
epsilon = 1e-6
ratios_analysis['total_assets_calc'] = (
    ratios_analysis['curr_assets'] + 
    ratios_analysis['fixed_assets'] + 
    ratios_analysis['intang_assets'] + 
    epsilon
)

ratios_analysis['fixed_asset_ratio'] = ratios_analysis['fixed_assets'] / ratios_analysis['total_assets_calc']
ratios_analysis['equity_ratio'] = ratios_analysis['share_eq'] / ratios_analysis['total_assets_calc']
ratios_analysis['liquidity_ratio'] = ratios_analysis['liq_assets'] / (ratios_analysis['curr_assets'] + epsilon)
ratios_analysis['inventory_ratio'] = ratios_analysis['inventories'] / (ratios_analysis['curr_assets'] + epsilon)
ratios_analysis['capitalization_ratio'] = ratios_analysis['subscribed_cap'] / (ratios_analysis['total_assets_calc'] + epsilon)

ratios_list = ['liquidity_ratio', 'inventory_ratio', 'fixed_asset_ratio', 'equity_ratio', 'capitalization_ratio']

for ratio in ratios_list:
    ratios_analysis[ratio] = ratios_analysis[ratio].clip(lower=-2.0, upper=2.0)

In [ ]:
ratios_analysis[['equity_ratio', 'capitalization_ratio']].corr()

In [ ]:

df_melted = ratios_analysis.melt(id_vars=['default'], value_vars=ratios_list, var_name='ratio', value_name='valor')

boxplot_final = alt.Chart(df_melted).mark_boxplot(
    extent='min-max',
    size=60,       
    median={'color': 'white', 'thickness': 2}
).encode(
    x=alt.X('default:N', title=None, axis=alt.Axis(labels=True)),
    y=alt.Y('valor:Q', title='Valor do Ratio'),
    color=alt.Color('default:N', 
                    scale=alt.Scale(domain=[0, 1], range=['#1f77b4', '#d62728']), 
                    legend=alt.Legend(title='Status do Default',
                                     orient='right',
                                     labelExpr="if(datum.label == '0', 'Não Default', 'Default')") 
                   )
).properties(
    width=200, 
    height=400
).facet(
    column=alt.Column('ratio:N', title='Análise de Ratios vs Default'),
    spacing=20
).resolve_scale(
    y='independent' 
).configure_view(
    stroke=None
)

boxplot_final.display()

**Equity ratio**, **fixed asset ratio** e **capitalization ratio** parecem variáveis com potencial. Inventory e Liquidity não dão fortes indicativos de que vão ajudar a nossa performance, por indicarem splits pouco consistentes para a métrica target.

In [ ]:
epsilon = 1e-6

df_clean_final_w_target['total_assets_aux'] = (
    df_clean_final_w_target['curr_assets'] + 
    df_clean_final_w_target['fixed_assets'] + 
    df_clean_final_w_target['intang_assets'] + 
    epsilon
)

df_clean_final_w_target['equity_ratio'] = (df_clean_final_w_target['share_eq'] / df_clean_final_w_target['total_assets_aux']).clip(0, 2)
df_clean_final_w_target['fixed_asset_ratio'] = (df_clean_final_w_target['fixed_assets'] / df_clean_final_w_target['total_assets_aux']).clip(0, 1)
df_clean_final_w_target['capitalization_ratio'] = (df_clean_final_w_target['subscribed_cap'] / df_clean_final_w_target['total_assets_aux']).clip(0, 1)


cols_to_drop = [
    'tang_assets', 
    'total_assets_aux'
]

df_clean_final_w_target = df_clean_final_w_target.drop(columns=cols_to_drop, errors='ignore')

## Last Year Sales Growth

Para identificar crescimento nas vendas, se apenas calcularmos o crescimento, empresas que não existiam em 2011 podem ser prejudicadas incorretamente. Vamos, além de criar a coluna de growth, incluir duas boleanas: `was_operating_2011` e `is_growing`, o que deveria ajudar os modelos a compreenderem a interação entre as métricas e segmentar com mais acurácia

In [ ]:
sales_metrics = sales_year[[2011, 2012]].copy()
sales_metrics

In [ ]:
sales_year.reset_index()

In [ ]:
sales_metrics = sales_year[[2011, 2012]].copy().reset_index()

sales_metrics['was_operating_2011'] = (sales_metrics[2011] > 0).astype(int)

sales_metrics['sales_growth'] = (sales_metrics[2012] - sales_metrics[2011]) / (sales_metrics[2011] + 1e-6)
sales_metrics['sales_growth'] = sales_metrics['sales_growth'].fillna(0)

# limpar eventuais outliers muito grandes
sales_metrics['sales_growth'] = sales_metrics['sales_growth'].clip(-1, 5) # Limita queda de 100% ou alta de 500%

sales_metrics.loc[sales_metrics['was_operating_2011'] == 0, 'sales_growth'] = 0
sales_metrics['is_growing'] = (sales_metrics['sales_growth'] > 0).astype(int)

In [ ]:
sales_metrics

In [ ]:
sales_metrics['comp_id'] = sales_metrics['comp_id'].astype(int)
df_clean_final_w_target['comp_id'] = df_clean_final_w_target['comp_id'].astype(int)

In [ ]:
df_clean_final_w_target = df_clean_final_w_target.merge(
    sales_metrics[['comp_id','was_operating_2011', 'sales_growth', 'is_growing']], 
    on='comp_id',      
    how='left'
)

## Variáveis categóricas — Agrupamento de categorias raras + One-Hot Encoding


Antes do `get_dummies`, categorias com frequência < 1% são agrupadas em `"other"`. Isso reduz drasticamente o número de colunas geradas sem perder as categorias relevantes.

As variáveis de baixa cardinalidade (`urban_m`, `region_m`) continuam com `get_dummies` direto.

In [ ]:
# Colunas de alta cardinalidade: agrupar categorias raras (< 1%) como 'other'
high_card_cols = ['nace_main', 'ind2']
for col in high_card_cols:
    freq = df_clean_final_w_target[col].value_counts(normalize=True)
    rare_cats = freq[freq < 0.01].index
    n_antes = df_clean_final_w_target[col].nunique()
    df_clean_final_w_target[col] = df_clean_final_w_target[col].where(~df_clean_final_w_target[col].isin(rare_cats), other='other')
    n_depois = df_clean_final_w_target[col].nunique()
    print(f'{col}: {n_antes} → {n_depois} categorias ({n_antes - n_depois} agrupadas em "other")')

# One-Hot Encoding em todas as categóricas
low_card_cols = ['urban_m', 'region_m']
cat_cols = high_card_cols + low_card_cols
df_final_with_dummies = pd.get_dummies(df_clean_final_w_target, columns=cat_cols, drop_first=True)
print(f'\nShape após dummies (v3): {df_final_with_dummies.shape}')

## Transformação Logarítmica

Variáveis financeiras costumam ter distribuição extremamente assimétrica (cauda longa à direita).
Calculamos o `skew` para identificar quais se beneficiam da transformação `log1p`.

Regra: skew > 5 → aplicar log.

In [ ]:
skew_values = df_final_with_dummies.select_dtypes(include='number').skew()

vars_high_skew = skew_values[skew_values > 5].index.tolist()

print(f"Total de variáveis com alta assimetria (> 5): {len(vars_high_skew)}")
print(vars_high_skew)

 Gráfico: Assimetria das variáveis numéricas

In [ ]:
skew_df = skew_values[skew_values > 5].reset_index()
skew_df.columns = ['variavel', 'skew']

alt.Chart(skew_df).mark_bar().encode(
    x=alt.X('skew:Q', title='Skewness'),
    y=alt.Y('variavel:N', sort='-x', title='Variável'),
    color=alt.condition(
        alt.datum.skew > 50,
        alt.value('crimson'),
        alt.value('orange')
    ),
    tooltip=['variavel', 'skew']
).properties(title='Variáveis com Skew > 5 (candidatas ao log)', width=500, height=350)

In [ ]:
df_2012_log = df_final_with_dummies.copy()

for col in vars_high_skew:
    if col in df_2012_log.columns:
        df_2012_log[f'log_{col}'] = np.log1p(df_2012_log[col].clip(lower=0))

# Remove as colunas originais (substituídas pelas versões log)
df_2012_log = df_2012_log.drop(
    columns=[col for col in vars_high_skew if col in df_2012_log.columns]
)

print(f'Shape após log transform: {df_2012_log.shape}')

In [ ]:
#Sanity Check: Logs
log_cols = [c for c in df_2012_log.columns if c.startswith('log_')]
df_2012_log[log_cols].skew().sort_values(ascending=False)

## Remoção de colunas com variância quase zero

Colunas onde mais de **98% dos valores são iguais ao modo** não contribuem para a modelagem — são dummies extremamente esparsas geradas pelas variáveis categóricas.

In [ ]:
nzv_cols = [
    col for col in df_2012_log.columns
    if col != 'default'
    and df_2012_log[col].value_counts(normalize=True).iloc[0] > 0.98
]

print(f'Colunas com variância quase zero removidas: {len(nzv_cols)}')
if nzv_cols:
    print('Exemplos:', nzv_cols[:10])

df_2012_log = df_2012_log.drop(columns=nzv_cols)

## Features Overview - pós feature engineering

In [ ]:
# Variáveis financeiras para análise bivariada
df_2012_log['default'] = df_2012_log['default'].astype(int)
vars_biv = ['log_sales', 'log_curr_assets', 'log_fixed_assets',
            'log_personnel_exp', 'log_amort', 'idade_empresa']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, var in enumerate(vars_biv):
    if var in df_2012_log.columns:
        sns.boxplot(
            data=df_2012_log,
            x='default',
            y=var,
            hue='default',
            palette=['steelblue', 'crimson'],
            ax=axes[i],
            legend=False
        )
        axes[i].set_title(f'{var} por Status')
        axes[i].set_xlabel('')
        axes[i].set_xticklabels(['Ativa', 'Deixou de operar'])

plt.suptitle('Análise Bivariada: Empresas Ativas vs Deixou de Operar', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Análise bivariada das demais variáveis transformadas durante o feature engineering

Além das variáveis financeiras principais analisadas acima, visualizamos a seguir as distribuições das demais variáveis log-transformadas e features engenheiradas, segmentadas por status de default.

In [ ]:
# --- Variáveis contínuas: Boxplots ---
# Separar variáveis esparsas (muitos zeros) das demais para melhor visualização
vars_sparse = ['log_extra_exp', 'log_extra_profit_loss', 'log_extra_inc', 'log_intang_assets']
vars_continuous = ['log_labor_avg', 'log_inventories', 'log_share_eq', 'log_subscribed_cap',
                   'log_liq_assets', 'log_material_exp', 'log_curr_liab',
                   'equity_ratio', 'fixed_asset_ratio', 'sales_growth']
vars_binary = ['was_operating_2011', 'is_growing']

# Filtrar apenas as que existem no dataframe
vars_sparse = [v for v in vars_sparse if v in df_2012_log.columns]
vars_continuous = [v for v in vars_continuous if v in df_2012_log.columns]
vars_binary = [v for v in vars_binary if v in df_2012_log.columns]

# --- 1) Boxplots das variáveis contínuas (não-esparsas) ---
if vars_continuous:
    n_cols = 3
    n_rows = (len(vars_continuous) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = axes.flatten()
    for i, var in enumerate(vars_continuous):
        sns.boxplot(data=df_2012_log, x='default', y=var, hue='default',
                    palette=['steelblue', 'crimson'], ax=axes[i], legend=False)
        axes[i].set_title(f'{var} por Status')
        axes[i].set_xlabel('')
        axes[i].set_xticklabels(['Ativa', 'Deixou de operar'])
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Análise Bivariada: Variáveis Contínuas Transformadas', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# --- 2) Variáveis esparsas: boxplot apenas dos valores > 0 ---
if vars_sparse:
    n_cols = 2
    n_rows = (len(vars_sparse) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows))
    axes = axes.flatten()
    for i, var in enumerate(vars_sparse):
        df_nonzero = df_2012_log[df_2012_log[var] > 0]
        pct_zero = (df_2012_log[var] == 0).mean() * 100
        sns.boxplot(data=df_nonzero, x='default', y=var, hue='default',
                    palette=['steelblue', 'crimson'], ax=axes[i], legend=False)
        axes[i].set_title(f'{var} por Status (somente > 0; {pct_zero:.0f}% são zero)')
        axes[i].set_xlabel('')
        axes[i].set_xticklabels(['Ativa', 'Deixou de operar'])
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Variáveis Esparsas: Distribuição entre valores não-zero', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# --- 3) Variáveis binárias: barplot com taxa de default ---
if vars_binary:
    fig, axes = plt.subplots(1, len(vars_binary), figsize=(6 * len(vars_binary), 5))
    if len(vars_binary) == 1:
        axes = [axes]
    for i, var in enumerate(vars_binary):
        ct = df_2012_log.groupby(var)['default'].mean().reset_index()
        ct.columns = [var, 'taxa_default']
        sns.barplot(data=ct, x=var, y='taxa_default',
                    palette=['steelblue', 'crimson'], ax=axes[i])
        axes[i].set_title(f'Taxa de Default por {var}')
        axes[i].set_ylabel('Taxa de Default')
        axes[i].set_xlabel(var)
        for bar in axes[i].patches:
            axes[i].annotate(f'{bar.get_height():.1%}',
                           (bar.get_x() + bar.get_width() / 2., bar.get_height()),
                           ha='center', va='bottom', fontsize=11)
    plt.suptitle('Variáveis Binárias: Taxa de Default por Categoria', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
vars_pair = ['log_sales', 'log_curr_assets', 'log_fixed_assets', 'log_personnel_exp', 'idade_empresa']

sns.pairplot(
    df_2012_log[vars_pair + ['default']].sample(1000, random_state=42),
    hue='default',
    palette={0: 'steelblue', 1: 'crimson'},
    diag_kind='kde',
    plot_kws={'alpha': 0.4}
)
plt.suptitle('Pairplot: Análise Exploratória por Status', y=1.02)
plt.show()

# Dataset Final

Verificação final do dataset pronto para modelagem.

In [ ]:
cols_excluir = ['comp_id', 'year', 'exit_']

df_final = df_2012_log.drop(columns=[c for c in cols_excluir if c in df_2012_log.columns])

print(f'Shape final: {df_final.shape}')
print(f'Missing values: {df_final.isnull().sum().sum()}')
print(f'Colunas object (devem ser 0): {df_final.select_dtypes("object").shape[1]}')
print(f'\nDistribuição do target:')
print(df_final['default'].value_counts())
print(f'Taxa de default: {df_final["default"].mean():.1%}')
df_final.head()

## Exportação do dataset pré-processado

Exportamos o dataset final em CSV, pronto para ser utilizado na etapa de modelagem.

O arquivo `bisnode_2012_preprocessado.csv` já inclui todas as limpezas e tratamentos dos dados:

- **Limpeza de dados:** Remoção de empresas com dados inconsistentes e tratamento de valores ausentes (missing values).
- **Variável target:** Criação da variável `default` a partir de sales, indicando se a empresa deixou de operar quando não houve vendas nos 2 anos subsequentes;

O notebook R pode carregar este CSV e iniciar diretamente o split treino/teste, sem nenhum tratamento adicional de dados.

In [ ]:
df_final.to_csv('bisnode_2012_preprocessado_vf.csv', index=False)
print(f'Dataset exportado: bisnode_2012_preprocessado_vf.csv')
print(f'Shape: {df_final.shape[0]} empresas x {df_final.shape[1]} variáveis')